# Classification + Class Imbalance + Cross-Validation

## Requirements
- Classification-models
- Cross-validation (Stratifies K-fold)
- Class imbalance handling (class_weight='balanced')
- Evaluation: Accuracy, Recall, Precision, F1, ROC AUC
- Threshold tuning support

In [1]:
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
%matplotlib inline

In [2]:
df=pd.read_csv('data/processed_heart_data.csv')
df.head()

,General_Health,Checkup,Exercise,Heart_Disease,Skin_Cancer,Other_Cancer,Depression,Diabetes,Arthritis,Sex,...,Alcohol_Consumption,Fruit_Consumption,Green_Vegetables_Consumption,FriedPotato_Consumption,Weight_(kg)_log,BMI_log,Alcohol_Consumption_log,Fruit_Consumption_log,Green_Vegetables_Consumption_log,FriedPotato_Consumption_log
0,0.0,3.0,0,0,0,0,0,0,1,0,...,-0.621527,0.006625,0.059597,0.664502,3.516310,-3.028379,-0.910061,0.393307,0.469585,1.083238
1,3.0,4.0,0,1,0,0,0,3,0,0,...,-0.621527,0.006625,-1.012342,-0.267579,4.358118,0.051097,-0.910061,0.393307,-2.315261,0.060136
2,3.0,4.0,1,0,0,0,0,3,0,0,...,-0.133707,-0.716973,-0.811354,1.130543,4.493680,0.842276,0.464207,-0.471177,-0.952633,1.370478
3,0.0,4.0,1,1,0,0,0,3,0,1,...,-0.621527,0.006625,0.997544,0.198462,4.547965,0.123540,-0.910061,0.393307,1.060102,0.689501
4,2.0,4.0,0,0,0,0,0,0,0,1,...,-0.621527,-0.877772,-0.744358,-0.733620,4.493680,-0.646970,-0.910061,-0.836975,-0.733299,-1.663148


In [24]:
!python -c "import sys; print(sys.executable)"

C:\ProgramData\miniconda3\python.exe


In [26]:
!pip show imbalanced-learn

Name: imbalanced-learn
Version: 0.14.0
Summary: Toolbox for imbalanced dataset in machine learning
Home-page: https://imbalanced-learn.org/
Author: 
Author-email: "G. Lemaitre" <g.lemaitre58@gmail.com>, "C. Aridas" <ichkoar@gmail.com>
License: 
Location: C:\Users\jsrri\AppData\Roaming\Python\Python313\site-packages
Requires: joblib, numpy, scikit-learn, scipy, threadpoolctl
Required-by: 


In [27]:
! python --version
! pip --version

Python 3.13.5
pip 25.2 from C:\Users\jsrri\AppData\Roaming\Python\Python313\site-packages\pip (python 3.13)



In [31]:
!C:\ProgramData\miniconda3\python.exe -m pip install imbalanced-learn

Defaulting to user installation because normal site-packages is not writeable


In [21]:
# !python -m pip install --upgrade pip setuptools wheel
# !python -m pip install imbalanced-learn

Defaulting to user installation because normal site-packages is not writeable
  Using cached pip-25.2-py3-none-any.whl.metadata (4.7 kB)
  Using cached setuptools-80.9.0-py3-none-any.whl.metadata (6.6 kB)
Using cached pip-25.2-py3-none-any.whl (1.8 MB)
Using cached setuptools-80.9.0-py3-none-any.whl (1.2 MB)

   ---------------------------------------- 0/2 [setuptools]
   ---------------------------------------- 0/2 [setuptools]
   ---------------------------------------- 0/2 [setuptools]
   ---------------------------------------- 0/2 [setuptools]
   ---------------------------------------- 0/2 [setuptools]
   ---------------------------------------- 0/2 [setuptools]
   ---------------------------------------- 0/2 [setuptools]
   ---------------------------------------- 0/2 [setuptools]
   ---------------------------------------- 0/2 [setuptools]
   ---------------------------------------- 0/2 [setuptools]
   ---------------------------------------- 0/2 [setuptools]
   ---------------

  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.


Defaulting to user installation because normal site-packages is not writeable
  Using cached joblib-1.5.2-py3-none-any.whl.metadata (5.6 kB)
  Using cached threadpoolctl-3.6.0-py3-none-any.whl.metadata (13 kB)
Using cached joblib-1.5.2-py3-none-any.whl (308 kB)
   ---------------------------------------- 0.0/8.7 MB ? eta -:--:--
   ------------------------ --------------- 5.2/8.7 MB 24.9 MB/s eta 0:00:01
   ---------------------------------------- 8.7/8.7 MB 21.4 MB/s  0:00:00
Using cached threadpoolctl-3.6.0-py3-none-any.whl (18 kB)

   ---------- ----------------------------- 1/4 [joblib]
   ---------- ----------------------------- 1/4 [joblib]
   ---------- ----------------------------- 1/4 [joblib]
   ---------- ----------------------------- 1/4 [joblib]
   ---------- ----------------------------- 1/4 [joblib]
   ---------- ----------------------------- 1/4 [joblib]
   ---------- ----------------------------- 1/4 [joblib]
   ---------- ----------------------------- 1/4 [joblib]
   

In [3]:
# Define model
# A mix of standard and advanced classifiers sitable for imbalance
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, AdaBoostClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.svm import SVC
from sklearn.naive_bayes import GaussianNB
from sklearn.neighbors import KNeighborsClassifier
from xgboost import XGBClassifier
from catboost import CatBoostClassifier

from sklearn.model_selection import StratifiedKFold, cross_val_score, train_test_split
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from imblearn.over_sampling import SMOTE


In [ ]:
models = {
    "Logistic Regression": LogisticRegression(class_weight='balanced', max_iter=500),
    "Random Forest": RandomForestClassifier(n_estimators=30, class_weight='balanced', n_jobs=-1, verbose=1),
    "XGBoost": XGBClassifier(n_estimators=30, use_label_encoder=False, eval_metric='logloss', n_jobs=-1, verbosity=1),
    "CatBoost": CatBoostClassifier(verbose=0, iterations=30),
    "AdaBoost": AdaBoostClassifier(n_estimators=30),
    "Naive Bayes": GaussianNB()  # Optional baseline
}

In [10]:
# Define evaluation function
# Use cross-validation and return multiple metrics
def evaluate_classifier(model, X, y, cv=3):
    skf = StratifiedKFold(n_splits=cv, shuffle=True, random_state=42)
    
    accuracy = cross_val_score(model, X, y, cv=skf, scoring='accuracy', n_jobs=-1)
    precision = cross_val_score(model, X, y, cv=skf, scoring='precision', n_jobs=-1)
    recall = cross_val_score(model, X, y, cv=skf, scoring='recall', n_jobs=-1)
    f1 = cross_val_score(model, X, y, cv=skf, scoring='f1', n_jobs=-1)
    roc_auc = cross_val_score(model, X, y, cv=skf, scoring='roc_auc', n_jobs=-1)
    
    return {
        "Accuracy": np.mean(accuracy),
        "Precision": np.mean(precision),
        "Recall": np.mean(recall),
        "F1 Score": np.mean(f1),
        "ROC AUC": np.mean(roc_auc)
    }


In [11]:
# Run all models and compare
import time
X = df.drop("Heart_Disease", axis=1)
y = df["Heart_Disease"]

model_names = []
results = []

for name, model in models.items():
    print(f"Evaluating: {name}")
    start = time.time()
    scores = evaluate_classifier(model, X, y)
    
    print(f"Accuracy: {scores['Accuracy']:.4f}")
    print(f"Precision: {scores['Precision']:.4f}")
    print(f"Recall: {scores['Recall']:.4f}")
    print(f"F1 Score: {scores['F1 Score']:.4f}")
    print(f"ROC AUC: {scores['ROC AUC']:.4f}")
     end = time.time()
    print(f"Total time taken: :{end - start: 2f} seconds")
    print("="*40)
    
    model_names.append(name)
    results.append(scores)
   


Evaluating: Logistic Regression
Accuracy: 0.7361
Precision: 0.2056
Recall: 0.7907
F1 Score: 0.3263
ROC AUC: 0.8352
Total time taken: : 16.527111 seconds
Evaluating: Random Forest
Accuracy: 0.9185
Precision: 0.4084
Recall: 0.0268
F1 Score: 0.0448
ROC AUC: 0.7866
Total time taken: : 68.979277 seconds
Evaluating: XGBoost
Accuracy: 0.9191
Precision: 0.4963
Recall: 0.0461
F1 Score: 0.0842
ROC AUC: 0.8346
Total time taken: : 12.686332 seconds
Evaluating: CatBoost


C:\Users\jsrri\AppData\Local\Programs\Python\Python312\Lib\site-packages\sklearn\model_selection\_validation.py:516: FitFailedWarning: 
2 fits failed out of a total of 3.
The score on these train-test partitions for these parameters will be set to nan.
If these failures are not expected, you can try to debug them by setting error_score='raise'.

Below are more details about the failures:
--------------------------------------------------------------------------------
1 fits failed with the following error:
Traceback (most recent call last):
  File "C:\Users\jsrri\AppData\Local\Programs\Python\Python312\Lib\site-packages\sklearn\model_selection\_validation.py", line 859, in _fit_and_score
    estimator.fit(X_train, y_train, **fit_params)
  File "C:\Users\jsrri\AppData\Local\Programs\Python\Python312\Lib\site-packages\catboost\core.py", line 5245, in fit
    self._fit(X, y, cat_features, text_features, embedding_features, None, graph, sample_weight, None, None, None, None, baseline, use_

Accuracy: nan
Precision: 0.5088
Recall: 0.0390
F1 Score: 0.0724
ROC AUC: 0.8356
Total time taken: : 16.995116 seconds
Evaluating: AdaBoost
Accuracy: 0.9188
Precision: 0.4871
Recall: 0.0679
F1 Score: 0.1190
ROC AUC: 0.8302
Total time taken: : 69.202829 seconds
Evaluating: Naive Bayes
Accuracy: 0.8245
Precision: 0.2310
Recall: 0.5025
F1 Score: 0.3165
ROC AUC: 0.7937
Total time taken: : 5.052636 seconds


In [12]:
# Create a dataframe for all model result
import pandas as pd

results_df = pd.DataFrame(results, index=model_names)
results_df = results_df.sort_values(by="ROC AUC", ascending=False)
print(results_df)


                     Accuracy  Precision    Recall  F1 Score   ROC AUC
CatBoost                  NaN   0.508798  0.038965  0.072376  0.835601
Logistic Regression  0.736066   0.205591  0.790677  0.326330  0.835223
XGBoost              0.919075   0.496263  0.046053  0.084245  0.834600
AdaBoost             0.918822   0.487130  0.067919  0.119029  0.830233
Naive Bayes          0.824532   0.231006  0.502503  0.316509  0.793724
Random Forest        0.918479   0.408403  0.026751  0.044820  0.786605


| Model                   | ROC AUC | Recall   | Precision | Comments                                       |
| ----------------------- | ------- | -------- | --------- | ---------------------------------------------- |
| **CatBoost**            | 0.8356  | 0.0390   | 0.5088    | High AUC but `NaN` accuracy → need to check data input |
| **Logistic Regression** | 0.8352  | **0.79** | 0.2056    | **Excellent recall**                           |
| XGBoost                 | 0.8346  | 0.0460   | 0.4963    | High AUC, low recall                           |
| AdaBoost                | 0.8302  | 0.0679   | 0.4871    | Similar to XGB                                 |
| Naive Bayes             | 0.7937  | 0.5025   | 0.2310    | Surprisingly good baseline                     |
| Random Forest           | 0.7866  | 0.0267   | 0.4084    | Not usable – too low recall                    |


## Best candidate for further work
- Logistic Regression - high recall, high AUC
- XGBoost/CatBoost - high AUC, potential with threshold tuning
- Naive Bayes - good fallback baseline

In [15]:
print(X.dtypes)

General_Health                      float64
Checkup                             float64
Exercise                              int64
Skin_Cancer                           int64
Other_Cancer                          int64
Depression                            int64
Diabetes                              int64
Arthritis                             int64
Sex                                   int64
Age_Category                        float64
Height_(cm)                         float64
Weight_(kg)                         float64
BMI                                 float64
Smoking_History                       int64
Alcohol_Consumption                 float64
Fruit_Consumption                   float64
Green_Vegetables_Consumption        float64
FriedPotato_Consumption             float64
Weight_(kg)_log                     float64
BMI_log                             float64
Alcohol_Consumption_log             float64
Fruit_Consumption_log               float64
Green_Vegetables_Consumption_log

In [16]:
print(X.dtypes.unique())

[dtype('float64') dtype('int64')]


In [17]:

# For testing, confirm all numeric
print("All numeric columns:", X.dtypes.unique())

# -------------------- 2. Train-Test Split --------------------
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=42
)

# -------------------- 3. Optional: SMOTE --------------------
apply_smote = False  # Toggle this

if apply_smote:
    smote = SMOTE(random_state=42)
    X_train, y_train = smote.fit_resample(X_train, y_train)
    print("SMOTE applied. Class distribution:")
    print(pd.Series(y_train).value_counts())

# -------------------- 4. Define Models --------------------
models = {
    "Logistic Regression": LogisticRegression(class_weight='balanced', max_iter=500),
    "Random Forest": RandomForestClassifier(n_estimators=30, class_weight='balanced', n_jobs=-1, verbose=0),
    "XGBoost": XGBClassifier(n_estimators=30, use_label_encoder=False, eval_metric='logloss', n_jobs=-1),
    "AdaBoost": AdaBoostClassifier(n_estimators=30),
    "Naive Bayes": GaussianNB(),

    # CatBoost with dedicated pipeline
    "CatBoost": Pipeline([
        ("scaler", StandardScaler()),  # Optional, helps other models in pipeline if needed
        ("catboost", CatBoostClassifier(verbose=0, iterations=30, random_state=42))
    ])
}

# -------------------- 5. Evaluation Function --------------------
def evaluate_classifier(model, X, y, cv=3):
    skf = StratifiedKFold(n_splits=cv, shuffle=True, random_state=42)
    return {
        "Accuracy": cross_val_score(model, X, y, cv=skf, scoring='accuracy', n_jobs=-1).mean(),
        "Precision": cross_val_score(model, X, y, cv=skf, scoring='precision', n_jobs=-1).mean(),
        "Recall": cross_val_score(model, X, y, cv=skf, scoring='recall', n_jobs=-1).mean(),
        "F1 Score": cross_val_score(model, X, y, cv=skf, scoring='f1', n_jobs=-1).mean(),
        "ROC AUC": cross_val_score(model, X, y, cv=skf, scoring='roc_auc', n_jobs=-1).mean()
    }

# -------------------- 6. Run All Models --------------------
results = []
model_names = []

for name, model in models.items():
    print(f"\n Evaluating: {name}")
    start = time.time()
    scores = evaluate_classifier(model, X, y)
    end = time.time()
    
    for metric, value in scores.items():
        print(f"{metric}: {value:.4f}")
    print(f"⏱ Time taken: {end - start:.2f} seconds")
    print("="*50)

    results.append(scores)
    model_names.append(name)

# -------------------- 7. Compile Results --------------------
results_df = pd.DataFrame(results, index=model_names).sort_values(by="ROC AUC", ascending=False)
print("\n Final Model Comparison:")
print(results_df)

# -------------------- 8. Threshold Tuning (for selected models) --------------------
def threshold_tuning(model, X_train, y_train, X_test, y_test, thresholds=np.arange(0.1, 0.9, 0.05)):
    model.fit(X_train, y_train)
    y_proba = model.predict_proba(X_test)[:, 1]

    print("\n Threshold Tuning Results:")
    for thresh in thresholds:
        y_pred_thresh = (y_proba >= thresh).astype(int)
        precision = precision_score(y_test, y_pred_thresh)
        recall = recall_score(y_test, y_pred_thresh)
        f1 = f1_score(y_test, y_pred_thresh)
        print(f"Threshold: {thresh:.2f} | Precision: {precision:.4f} | Recall: {recall:.4f} | F1: {f1:.4f}")

# Apply on top models only
print("\n================= Threshold Tuning: CatBoost =================")
threshold_tuning(models["CatBoost"], X_train, y_train, X_test, y_test)

print("\n================= Threshold Tuning: Logistic Regression =================")
threshold_tuning(models["Logistic Regression"], X_train, y_train, X_test, y_test)


All numeric columns: [dtype('float64') dtype('int64')]


NameError: name 'train_test_split' is not defined